In [3]:
import pandas as pd
import numpy as np

In [28]:
txt = pd.read_csv('/content/harry_potter_books.txt')

In [5]:
txt.sample(10)

,text,book,chapter
19939,did he run?` Harry had been wondering the sam...,Book 3: Prisoner of Azkaban,chap-14
51641,"Hall, where they could hear nothing except the...",Book 5: Order of the Phoenix,chap-19
52974,"meetings, haven't we?' 'You know what I mea...",Book 5: Order of the Phoenix,chap-21
3611,"know about it, Weasley, you couldn't afford ha...",Book 1: Philosopher's Stone,chap-10
4480,this had been his father's. He let the materia...,Book 1: Philosopher's Stone,chap-12
67238,boxes he was still clutching as he fumbled wit...,Book 6: Half Blood Prince,chap-6
46933,"Harry loudly, his fist in the air again. Pr...",Book 5: Order of the Phoenix,chap-12
20316,"certainly did,` said Snape, his face contorted...",Book 3: Prisoner of Azkaban,chap-14
74081,"bowed low over Hepzibah's fat little hand, bru...",Book 6: Half Blood Prince,chap-20
72043,"tree. Finally, he decided on the truth ... or ...",Book 6: Half Blood Prince,chap-16


In [6]:
len(txt)

95085

In [29]:
X = txt['text']
X

,text
0,"THE BOY WHO LIVED Mr. and Mrs. Dursley, of nu..."
1,"four, Privet Drive, were proud to say that the..."
2,thank you very much. They were the last people...
3,"be involved in anything strange or mysterious,..."
4,with such nonsense. Mr. Dursley was the direc...
...,...
95080,son glide away from him....The last trace of s...
95081,autumn air. The train rounded a corner. Harry'...
95082,"in farewell.`He'll be alright,` murmured Ginny..."
95083,his hand absentmindedly and touched the lightn...


In [30]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [31]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X)

In [32]:
len(tokenizer.word_index)

25134

In [33]:
input_sequences = []
for sentence in X:
  tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

  for i in range(1,len(tokenized_sentence)):
    input_sequences.append(tokenized_sentence[:i+1])

In [12]:
input_sequences

[[1, 269],
 [1, 269, 46],
 [1, 269, 46, 1310],
 [1, 269, 46, 1310, 121],
 [1, 269, 46, 1310, 121, 2],
 [1, 269, 46, 1310, 121, 2, 169],
 [1, 269, 46, 1310, 121, 2, 169, 1360],
 [1, 269, 46, 1310, 121, 2, 169, 1360, 4],
 [1, 269, 46, 1310, 121, 2, 169, 1360, 4, 641],
 [441, 1351],
 [441, 1351, 1044],
 [441, 1351, 1044, 35],
 [441, 1351, 1044, 35, 1913],
 [441, 1351, 1044, 35, 1913, 3],
 [441, 1351, 1044, 35, 1913, 3, 168],
 [441, 1351, 1044, 35, 1913, 3, 168, 16],
 [441, 1351, 1044, 35, 1913, 3, 168, 16, 23],
 [441, 1351, 1044, 35, 1913, 3, 168, 16, 23, 35],
 [441, 1351, 1044, 35, 1913, 3, 168, 16, 23, 35, 1209],
 [441, 1351, 1044, 35, 1913, 3, 168, 16, 23, 35, 1209, 1105],
 [795, 13],
 [795, 13, 74],
 [795, 13, 74, 146],
 [795, 13, 74, 146, 23],
 [795, 13, 74, 146, 23, 35],
 [795, 13, 74, 146, 23, 35, 1],
 [795, 13, 74, 146, 23, 35, 1, 141],
 [795, 13, 74, 146, 23, 35, 1, 141, 164],
 [795, 13, 74, 146, 23, 35, 1, 141, 164, 496],
 [795, 13, 74, 146, 23, 35, 1, 141, 164, 496, 840],
 [795

In [34]:
max_len = max([len(x) for x in input_sequences])
max_len

38

In [35]:
padded_input_sequences = pad_sequences(input_sequences, maxlen = max_len, padding='pre')

In [15]:
padded_input_sequences

array([[   0,    0,    0, ...,    0,    1,  269],
       [   0,    0,    0, ...,    1,  269,   46],
       [   0,    0,    0, ...,  269,   46, 1310],
       ...,
       [   0,    0,    0, ..., 7407,  268,   34],
       [   0,    0,    0, ...,  268,   34,    8],
       [   0,    0,    0, ...,   34,    8,   92]], dtype=int32)

In [36]:
X = padded_input_sequences[:,:-1]
y = padded_input_sequences[:,-1]

In [37]:
X.shape,y.shape

((1006186, 37), (1006186,))

In [38]:
from tensorflow.keras.utils import to_categorical
# Reverting y to integer labels instead of one-hot encoding to save RAM
# The original y before to_categorical was the integer labels
y = padded_input_sequences[:,-1]

In [47]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense

model = Sequential()
model.add(Embedding(25135, 128, input_shape=(38,)))
model.add(GRU(150))
model.add(Dense(25135, activation='softmax'))

In [48]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam',metrics=['accuracy'])

In [49]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 38, 128)        │     3,217,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ (None, 150)            │       126,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 25135)          │     3,795,385 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,138,665 (27.23 MB)

 Trainable params: 7,138,665 (27.23 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
history = model.fit(X,y,epochs=10)

Epoch 1/10
31444/31444 ━━━━━━━━━━━━━━━━━━━━ 369s 12ms/step - accuracy: 0.1326 - loss: 5.8736
Epoch 2/10
31444/31444 ━━━━━━━━━━━━━━━━━━━━ 361s 11ms/step - accuracy: 0.1701 - loss: 5.3587
Epoch 3/10
31444/31444 ━━━━━━━━━━━━━━━━━━━━ 363s 12ms/step - accuracy: 0.1861 - loss: 5.1384
Epoch 4/10
31444/31444 ━━━━━━━━━━━━━━━━━━━━ 363s 12ms/step - accuracy: 0.1986 - loss: 4.9615
Epoch 5/10
31444/31444 ━━━━━━━━━━━━━━━━━━━━ 364s 12ms/step - accuracy: 0.2088 - loss: 4.8124
Epoch 6/10
31444/31444 ━━━━━━━━━━━━━━━━━━━━ 364s 12ms/step - accuracy: 0.2176 - loss: 4.6915
Epoch 7/10
31444/31444 ━━━━━━━━━━━━━━━━━━━━ 362s 12ms/step - accuracy: 0.2249 - loss: 4.5912
Epoch 8/10
31444/31444 ━━━━━━━━━━━━━━━━━━━━ 363s 12ms/step - accuracy: 0.2313 - loss: 4.5113
Epoch 9/10
31444/31444 ━━━━━━━━━━━━━━━━━━━━ 362s 12ms/step - accuracy: 0.2370 - loss: 4.4488
Epoch 10/10
31444/31444 ━━━━━━━━━━━━━━━━━━━━ 362s 12ms/step - accuracy: 0.2413 - loss: 4.3992


In [27]:
import numpy as np
import time
text = "harry"

for i in range(10):
  # tokenize
  token_text = tokenizer.texts_to_sequences([text])[0]
  # padding
  padded_token_text = pad_sequences([token_text], maxlen=56, padding='pre')
  # predict
  pos = np.argmax(model.predict(padded_token_text))

  for word,index in tokenizer.word_index.items():
    if index == pos:
      text = text + " " + word
      print(text)
      time.sleep(2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
harry and
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
harry and ron
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
harry and ron were
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
harry and ron were both
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
harry and ron were both their
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
harry and ron were both their feet
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
harry and ron were both their feet and
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
harry and ron were both their feet and the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
harry and ron were both their feet and the dead
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
harry and ron were both their feet and the dead of
